In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_pl_2024_06.csv
/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_rent_pl_2024_03.csv
/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_rent_pl_2024_02.csv
/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_pl_2024_02.csv
/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_rent_pl_2023_12.csv
/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_pl_2023_12.csv
/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_rent_pl_2024_05.csv
/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_rent_pl_2024_06.csv
/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_rent_pl_2024_04.csv
/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_pl_2024_04.csv
/kaggle/input/datasets/krzysztofjamroz/apartment-p

In [2]:
import glob

# Load and combine all sale files
files = glob.glob('/kaggle/input/datasets/krzysztofjamroz/apartment-prices-in-poland/apartments_pl_*.csv')
data = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

In [3]:
# Fill missing categoricals with 'unknown'
data['condition'] = data['condition'].fillna('unknown')
data['type'] = data['type'].fillna('unknown')
data['buildingMaterial'] = data['buildingMaterial'].fillna('unknown')

# Fill missing numericals
data['floor'] = data['floor'].fillna(data['floor'].median())
data['floorCount'] = data['floorCount'].fillna(data['floorCount'].median())
data['buildYear'] = data['buildYear'].fillna(data['buildYear'].median())

# Engineer age feature
data['age'] = 2024 - data['buildYear']

In [4]:
data = data.drop(columns=['id', 'buildYear', 'latitude', 'longitude',
                       'schoolDistance', 'clinicDistance', 'postOfficeDistance',
                       'kindergartenDistance', 'restaurantDistance', 
                       'collegeDistance', 'pharmacyDistance'])

In [5]:
data['hasElevator'] = data['hasElevator'].fillna(data['hasElevator'].mode()[0])
print(data.isnull().sum().sum())  # should print 0

0


In [6]:
bool_cols = ['hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity', 'hasStorageRoom']

for col in bool_cols:
    data[col] = (data[col] == 'yes').astype(int)

In [7]:
data['amenityScore'] = (data['hasParkingSpace'] + data['hasBalcony'] + 
                      data['hasElevator'] + data['hasSecurity'] + 
                      data['hasStorageRoom'])

In [8]:
data = data.drop(columns=['hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity', 'hasStorageRoom'])

In [9]:
data['centreDistance'] = np.log(data['centreDistance'] + 1)
data['age'] = np.log(data['age'] + 1)
data['floor'] = np.log(data['floor'] + 1)
data['squareMeters'] = np.log(data['squareMeters'] + 1)
data['poiCount'] = np.log(data['poiCount'] + 1)
data['floorCount'] = np.log(data['floorCount'] + 1)

In [10]:
# Do this before pd.get_dummies
data['cityCount'] = data.groupby('city')['city'].transform('count')

data['floorRatio'] = data['floor'] / data['floorCount'].replace(0, 1)

In [11]:
# Encode categoricals
data = pd.get_dummies(data, columns=['city', 'type', 'buildingMaterial', 'condition', 'ownership'], drop_first=True)

# Define X and y
X = data.drop(['price'], axis=1)
y = data['price']

In [12]:
city_cols = [col.replace('city_', '') for col in X_train.columns if col.startswith('city_')]
type_cols = [col.replace('type_', '') for col in X_train.columns if col.startswith('type_')]
material_cols = [col.replace('buildingMaterial_', '') for col in X_train.columns if col.startswith('buildingMaterial_')]
condition_cols = [col.replace('condition_', '') for col in X_train.columns if col.startswith('condition_')]
ownership_cols = [col.replace('ownership_', '') for col in X_train.columns if col.startswith('ownership_')]

print("Cities:", city_cols)
print("Types:", type_cols)
print("Materials:", material_cols)
print("Conditions:", condition_cols)
print("Ownership:", ownership_cols)

NameError: name 'X_train' is not defined

In [ ]:
all_cities = ['bialystok'] + [col.replace('city_', '') for col in X_train.columns if col.startswith('city_')]
all_ownership = [col.replace('ownership_', '') for col in X_train.columns if col.startswith('ownership_')]
print("All cities:", sorted(all_cities))
print("Ownership reference (dropped):", all_ownership)

In [ ]:
# Split
from sklearn.model_selection import train_test_split

y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.utils import resample

params = {
    'n_estimators': [300, 500, 1000],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 6, 8],
    'colsample_bytree': [0.7, 0.8],
    'subsample': [0.7, 0.8]
}

xgb_grid = XGBRegressor(
    random_state=42,
    device='cuda',        # enables GPU acceleration
    tree_method='hist'    # required for GPU
)

# Use 20% of data just for grid search — still 39k rows, plenty
X_sample, _, y_sample, _ = train_test_split(X_train, y_train, test_size=0.8, random_state=42)

grid_search = GridSearchCV(
    xgb_grid,
    params,
    cv=5,
    scoring='r2',
    verbose=1,
    n_jobs=1
)

grid_search.fit(X_sample, y_sample)

print("Best params:", grid_search.best_params_)
print("Best CV R²:", grid_search.best_score_.round(4))

In [ ]:
# Train
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor
xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    colsample_bytree=0.8,
    subsample=0.8,
    reg_alpha=0.1,
    reg_lambda=1.5,
    random_state=42
)
xgb.fit(X_train, y_train)

preds_log = xgb.predict(X_test)
preds = np.expm1(preds_log)
y_test_actual = np.expm1(y_test)

print("Train R²:", xgb.score(X_train, y_train))
print("Test R²:", xgb.score(X_test, y_test))  # R² on log scale
print("MAE:", f"{mean_absolute_error(y_test_actual, preds):,.0f} zł")
print("RMSE:", f"{root_mean_squared_error(y_test_actual, preds):,.0f} zł")

In [ ]:
print(X_train['ownership'].value_counts())
print(X_train['city'].unique().tolist())

In [ ]:
import joblib

joblib.dump(xgb, 'xgb_model_polish.pkl')
joblib.dump(X_train.columns.tolist(), 'model_columns_polish.pkl')

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(xgb, X, y, cv=5, scoring='r2')
print("CV scores:", scores)
print("Mean R²:", scores.mean().round(4))
print("Std:", scores.std().round(4))